This tutorial shows the functionality of workgroups in OpenCL.

In [ ]:
!pip install pyopencl

In [ ]:
import numpy as np
import pyopencl as cl
import pyopencl.array as cl_array

OpenCL always needs a ```context``` object, which stores information con the programming environment.

Also, a ````queue```` is needed to store the sequence of instructions.

In [ ]:
ctx = cl.create_some_context()
queue = cl.CommandQueue(ctx)

OpenCL divides the data in work groups. Each work group contains many threads to perform the same operation but on different data (SIMT model). There is a maximum size of each working group, which depends on the compute device.

In [ ]:
platform = cl.get_platforms()[0]
device = platform.get_devices()[0]
print("Platform name:", platform.name)
print("Device name:", device.name)
print("Maximum work group size:", device.max_work_group_size)

The code that needs to be executed by OpenCL is called a 'kernel'. The kernel needs to be written as a piece of C code, where one can use the functionality of OpenCL as well. This piece of C code is stored in a text string and will be compiled by the program. Each function in the C code needs to start with the prefix ```___kernel```.

In OpenCL, the code is written for a single thread. That is, one needs to program the instructions for a single thread and OpenCL will asign threads to the data array.

The 'local ID' is the location of the thread in the workgroup.

The 'group ID' is the workgroup to which the thread belongs.

The 'global ID' is the location of the data point in the data array.

The input variable '0' of these functions refer to the first dimension of the grid. In OpenCL, you can use 2D and 3D grids as well.

In [ ]:
kernel = """
__kernel void get_id(__global int *vec1,
                     __global int *vec2,
                     __global int *vec3,
                     __global int *vec4,
                     __global int *vec5,
                     __global int *vec6)
{
  int id = get_global_id(0);

  vec1[id] = get_global_id(0);
  vec2[id] = get_group_id(0);
  vec3[id] = get_local_id(0);
  vec4[id] = get_global_size(0);
  vec5[id] = get_num_groups(0);
  vec6[id] = get_local_size(0);
}
"""

The kernel needs to be compiled to build a program.

In [ ]:
prg = cl.Program(ctx, kernel).build()

Now, let us specify the size of the workgroup and the number of workgroups we wish to use later on.

In [ ]:
workgroup_size = 6
n_workgroups = 4
n_vector = workgroup_size * n_workgroups

Let us create four OpenCL arrays that will store the IDs of each thread.

In [ ]:
cl_global_id = cl_array.empty(queue, n_vector, dtype=np.int32)
cl_group_id = cl_array.empty(queue, n_vector, dtype=np.int32)
cl_local_id = cl_array.empty(queue, n_vector, dtype=np.int32)
cl_global_size = cl_array.empty(queue, n_vector, dtype=np.int32)
cl_num_groups = cl_array.empty(queue, n_vector, dtype=np.int32)
cl_local_size = cl_array.empty(queue, n_vector, dtype=np.int32)

With both the program and data available, we can execute the kernel on the compute device. For this, we will create an 'event' by specifying which function in the kernel we wish to execute and providing the input variables.

The input variables of the program are as follows:
 1. the command queue
 1. the global size of the data array, possibly multi-dimensional
 1. the size of the workgroups, which needs to divide the global size evenly
 1. the buffers for the input variables of the kernel function

In [ ]:
event = prg.get_id(queue,
                   (n_vector,),
                   (workgroup_size,),
                   cl_global_id.data,
                   cl_group_id.data,
                   cl_local_id.data,
                   cl_global_size.data,
                   cl_num_groups.data,
                   cl_local_size.data)

In [ ]:
print("Global ID:")
print(cl_global_id)
print("Group ID:")
print(cl_group_id)
print("Local ID:")
print(cl_local_id)
print("Global size:")
print(cl_global_size)
print("Number of groups:")
print(cl_num_groups)
print("Local size:")
print(cl_local_size)